In [1]:
import os, re
import numpy as np
import warnings 
import pandas as pd 

from typing import Dict, List, Optional

from datetime import datetime
import xarray as xr
import xskillscore as xs

from xcdat.dataset import open_dataset
from xcdat.bounds import create_bounds
from xcdat.dataset import open_mfdataset
from pathlib import Path
from typing import Sequence, Optional

from xarray.conventions import SerializationWarning
#warnings.filterwarnings("ignore", category=SerializationWarning)
warnings.filterwarnings("once", category=SerializationWarning)

/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/esmpy/interface/loadESMF.py:94: VersionWarning: ESMF installation version 8.8.0, ESMPy version 8.8.0b0
  warnings.warn("ESMF installation version {}, ESMPy version {}".format(


In [2]:
def extract_exp_info(
    data_path: str,
    *,
    resolution: str = "ne30pg2_r05_IcoswISC30E3r5",
    machine: str = "compy",
    atm_subdir: str = "archive/post/atm/180x360_aave",
    lnd_subdir: str = "archive/post/lnd/180x360_aave",
) -> Dict[str, dict]:
    """
    Build a standardized experiment metadata dictionary for E3SM ensemble and DA runs.

    Returns a dict keyed by experiment name with:
      - nens, season, group_key
      - runs: {da|fc|wc} -> sub-run dict or None
      - default_run, key, period
    """

    exps = {
        "CTRL": {
            "nens": 1,
            "key": "ctrl",
            "da_run": None,
            "fc_run": {"compset": "F20TR", "name": "CTRL", "period": "201201-201212"},
            "wc_run": {"compset": "WCYCL20TR", "name": "CTRL", "period": "201201-201212"},
        },
        "CTRL10-S0": {
            "nens": 10,
            "key": "dart_en10",
            "da_run": {
                "compset": "F20TR", "name": "CTRLEN10", "period": "201112-201112",
                "obs_diag": ["obs_seq", "obs_diag", "obs_common", "closest_member"],
            },
            "fc_run": {"compset": "F20TR", "name": "CTRLEN10_15day", "period": "201201-201202"},
            "wc_run": {"compset": "WCYCL20TR", "name": "CTRLEN10_15day", "period": "201201-201202"},
        },
        "CAPT10-S0": {
            "nens": 10,
            "key": "dart_en10",
            "da_run": None,
            "fc_run": {"compset": "F20TR", "name": "CAPTEN10_15day", "period": "201201-201202"},
            "wc_run": {"compset": "WCYCL20TR", "name": "CAPTEN10_15day", "period": "201201-201202"},
        },
        "DART10-S0": {
            "nens": 10,
            "key": "dart_en10",
            "da_run": {
                "compset": "F20TR", "name": "DARTEN10", "period": "201112-201112",
                "obs_diag": ["obs_seq", "obs_diag", "obs_common", "closest_member"],
            },
            "fc_run": None,
            "wc_run": None,
        },
        "DART20-S0": {
            "nens": 20,
            "key": "dart_en20",
            "da_run": {
                "compset": "F20TR", "name": "DARTEN20", "period": "201112-201112",
                "obs_diag": ["obs_seq", "obs_diag", "obs_common", "closest_member"],
            },
            "fc_run": {"compset": "F20TR", "name": "DARTEN20_15day", "period": "201201-201202"},
            "wc_run": {"compset": "WCYCL20TR", "name": "DARTEN20_15day", "period": "201201-201202"},
        },
        "DART40-S0": {
            "nens": 40,
            "key": "dart_en40",
            "da_run": {
                "compset": "F20TR", "name": "DARTEN40", "period": "201112-201112",
                "obs_diag": ["obs_seq", "obs_diag", "obs_common", "closest_member"],
            },
            "fc_run": {"compset": "F20TR", "name": "DARTEN40_15day", "period": "201201-201202"},
            "wc_run": {"compset": "WCYCL20TR", "name": "DARTEN40_15day", "period": "201201-201202"},
        },
        "DART40INF0p6-S0": {
            "nens": 40,
            "key": "dart_en40",
            # can treat 'alia' as documentation-only for now
            "da_run": {
                "compset": "F20TR", "name": "DARTEN40_INF0p6", "alia": "DARTEN40", "period": "201112-201112",
                "obs_diag": ["obs_seq", "obs_diag", "obs_common", "closest_member"],
            },
            "fc_run": None,
            "wc_run": None,
        },
        "CTRL10-S1": {
            "nens": 10,
            "key": "ctrl_en10",
            "da_run": None,
            "fc_run": {"compset": "F20TR", "name": "CTRLEN10s1_15day", "period": "201206-201207"},
            "wc_run": {"compset": "WCYCL20TR", "name": "CTRLEN10s1_15day", "period": "201206-201207"},
        },
        "CAPT10-S1": {
            "nens": 10,
            "key": "capt_en10",
            "da_run": None,
            "fc_run": {"compset": "F20TR", "name": "CAPTEN10S1_15day", "period": "201206-201207"},
            "wc_run": {"compset": "WCYCL20TR", "name": "CAPTEN10S1_15day", "period": "201206-201207"},
        },
        "DART40-S1": {
            "nens": 40,
            "key": "dart_en40",
            "da_run": {
                "compset": "F20TR", "name": "DARTEN40S1", "period": "201205-201205",
                "obs_diag": ["obs_seq", "obs_diag", "obs_common", "closest_member"],
            },
            "fc_run": {"compset": "F20TR", "name": "DARTEN40S1_15day", "period": "201206-201207"},
            "wc_run": {"compset": "WCYCL20TR", "name": "DARTEN40S1_15day", "period": "201206-201207"},
        },
    }

    def _season_from_name(name: str) -> Optional[str]:
        _SEASON_RE = re.compile(r"(?:-|_)(S\d+)\b")
        m = _SEASON_RE.search(name)
        return m.group(1) if m else None

    def _require_valid_period(p: str) -> str:
        _PERIOD_RE = re.compile(r"^\d{6}-\d{6}$")
        if not _PERIOD_RE.match(p):
            raise ValueError(f"Invalid period '{p}' for experiment; expected 'YYYYMM-YYYYMM'.")
        return p

    def _build_run(exp_name: str, spec: Optional[dict]) -> Optional[dict]:
        if spec is None:
            return None
        period = _require_valid_period(spec["period"])
        run_id = spec.get(
            "run_id",
            f"{spec['name']}_{spec['compset']}_{resolution}_{machine}",
        )
        atm_path = os.path.join(data_path, run_id, atm_subdir)
        lnd_path = os.path.join(data_path, run_id, lnd_subdir)
        out = {
            "run_id": run_id,
            "name": spec["name"],
            "compset": spec["compset"],
            "period": period,
            "atm": atm_subdir,
            "lnd": lnd_subdir,
            "atm_path": atm_path,
            "lnd_path": lnd_path,
        }
        if "obs_diag" in spec:
            out["obs_diag"] = list(spec["obs_diag"])
        return out

    exp_dict: Dict[str, dict] = {}
    for exp_name, meta in sorted(exps.items()):
        runs = {
            "da": _build_run(exp_name, meta.get("da_run")),
            "fc": _build_run(exp_name, meta.get("fc_run")),
            "wc": _build_run(exp_name, meta.get("wc_run")),
        }
        default_run = runs["fc"] or runs["wc"] or runs["da"]
        exp_dict[exp_name] = {
            "nens": meta["nens"],
            "season": _season_from_name(exp_name),
            "group_key": meta.get("key"),
            "runs": runs,
            "default_run": default_run,
            "key": meta.get("key"),
            "period": (default_run or {}).get("period"),
        }

    return exp_dict

In [3]:
class EnsembleMetricEvaluator:
    """
    Robust evaluator for ensemble monthly fields vs observations.

    Highlights
    ----------
    • Modern, deprecation-safe CF-time decoding (use_cftime='auto').
    • Year–month selection via .dt (avoids "not all values found in index 'time'").
    • Lat/lon normalization (lat|latitude, lon|longitude), optional regridding.
    • Missing-value unification (attrs + common sentinels).
    • Clear logging + options: allow_missing_months, enforce_grid_match, chunking.
    • Bootstrap CIs + significance masks for bias/RMSE/spread/CRPS maps.
    """

    # --------------------------- public runner ---------------------------
    @classmethod
    def run_one(
        cls,
        *,
        var_name: str,
        var_cfg: dict,
        freq: str,
        run_seg: str, 
        component: str, 
        exp_key: str,
        exp_meta: dict,
        obs_var_map: dict,
        model_label: str | None = None,
        obs_root: str,
        out_path: str,
        allow_missing_months: bool = True,
        enforce_grid_match: bool = False,
        chunks: dict | None = None,
        n_boot_map: int = 500,
        n_boot_scalar: int = 1000,
        alpha: float = 0.05,
    ):
        """
        Execute metrics for a single (variable, experiment) pair.
        """
        Path(out_path).mkdir(parents=True, exist_ok=True)
        
        # --- resolve experiment/run paths
        exp_cfg = None
        period = None

        if run_seg is not None and exp_meta.get("runs"):
            # expect run_seg in {"da","fc","wc"}
            run_dict = exp_meta["runs"]
            sel = run_dict.get(run_seg)
            if sel:
                exp_cfg = sel
                period = exp_cfg.get("period")
            else:
                print(f"[WARN] Skip {exp_key}: runs has no '{run_seg}' entry")
                return None
        elif exp_meta.get("default_run"):
            exp_cfg = exp_meta["default_run"]
            period = exp_cfg.get("period")
        else:
            print(f"[WARN] Skip {exp_key}: no runs/{run_seg or ''} and no default_run")
            return None

        if not period:
            print(f"[WARN] Skip {exp_key}: missing period for selected run")
            return None

        nens = int(exp_meta.get("nens", 0)) or 0
        if nens <= 0:
            print(f"[WARN] Skip {exp_key}: invalid nens={exp_meta.get('nens')}")
            return None

        exp_data_dir = cls.exp_data_path(exp_cfg, component=component, freq=freq)

        # --- resolve obs variable names & target units
        ref_product = var_cfg["ref"]
        target_unit = var_cfg["unit"]
        try_vars = obs_var_map.get(ref_product, {}).get(var_name, [var_name])

        # ---- observations
        try:
            obs_ds = cls.open_obs_series(
                obs_root, ref_product, period, try_vars=try_vars,
                allow_missing_months=allow_missing_months, chunks=chunks
            )
        except Exception as e:
            print(f"[WARN] Skip {exp_key}·{var_name}: OBS open failed ({ref_product}, {period}): {e}")
            return None

        obs_name = next((v for v in try_vars if v in obs_ds.data_vars), None)
        if obs_name is None:
            present = list(obs_ds.data_vars)
            print(f"[WARN] Skip {exp_key}·{var_name}: none of {try_vars} in OBS (have {present})")
            return None

        obs_da = cls.convert_units(obs_ds[obs_name], var_name, target_unit, product=ref_product)

        # ---- model
        try:
            mod_da = cls.open_model_cube(
                exp_data_dir, var_name, nens, period,
                allow_missing_months=allow_missing_months, chunks=chunks
            )
        except Exception as e:
            print(f"[WARN] Skip {exp_key}·{var_name}: MODEL open failed ({exp_data_dir}): {e}")
            return None

        mod_da = cls.convert_units(mod_da, var_name, target_unit)

        # Optional grid alignment
        if enforce_grid_match:
            try:
                mod_da = mod_da.interp(lat=obs_da.lat, lon=obs_da.lon)
            except Exception as e:
                print(f"[WARN] {exp_key}·{var_name}: regridding failed; continue on native grids: {e}")

        # ---- evaluate & save
        evaluator = cls(
            obs=obs_da,
            model_dict={exp_key: mod_da},
            component=component,
            ref_dataset=ref_product,
            var=var_name,
            vunit=target_unit,
            output_path=out_path,
            n_boot_map=n_boot_map,
            n_boot_scalar=n_boot_scalar,
            alpha=alpha,
        )

        # attach pretty label if provided
        if model_label:
            for _, bundle in evaluator.results.items():
                bundle.setdefault("metrics", {})["model_label"] = model_label

        print(f"[DONE] {model_label or exp_key} · {var_name} · {period}")
        return evaluator.results


    # ------------------------- static I/O helpers -------------------------

    @staticmethod
    def _parse_period_ym(period_str: str) -> tuple[int, int, int, int]:
        m = re.fullmatch(r"(\d{6})-(\d{6})", period_str)
        if not m:
            raise ValueError(f"Bad period: {period_str} (expected 'YYYYMM-YYYYMM')")
        s, e = m.groups()
        return int(s[:4]), int(s[-2:]), int(e[:4]), int(e[-2:])

    @staticmethod
    def exp_data_path(exp_cfg: dict, component: str = "atm", freq: str = "monthly") -> str:
        """
        Return absolute data directory for this run and frequency.
        exp_cfg is the selected run dict (e.g., runs['fc'] or default_run).
        """
        subkey = f"{component}_path"  # 'atm_path' or 'lnd_path'
        base = exp_cfg.get(subkey)
        if not base:
            raise ValueError(f"run config missing '{subkey}'")
        return f"{base}/{freq}"

    @staticmethod
    def _normalize_coords(ds: xr.Dataset | xr.DataArray) -> xr.Dataset | xr.DataArray:
        ren = {}
        if "latitude" in ds.coords:  ren["latitude"] = "lat"
        if "longitude" in ds.coords: ren["longitude"] = "lon"
        if ren:
            ds = ds.rename(ren)
        return ds

    @staticmethod
    def _select_period_by_ym(ds: xr.Dataset | xr.DataArray, y0: int, m0: int, y1: int, m1: int,
                             allow_missing_months: bool) -> xr.Dataset | xr.DataArray:
        ym = ds.time.dt.year * 100 + ds.time.dt.month
        lo, hi = y0 * 100 + m0, y1 * 100 + m1
        sel = ds.sel(time=(ym >= lo) & (ym <= hi))
        if not allow_missing_months:
            want = (y1 - y0) * 12 + (m1 - m0) + 1
            got = sel.sizes.get("time", 0)
            if got != want:
                raise ValueError(f"Missing months in selection: wanted {want}, got {got}")
        return sel

    @staticmethod
    def _cf_decode_opts() -> dict:
        """
        Return kwargs for xarray decode_times that work across xarray versions.
        Newer xarray: pass a CFDatetimeCoder instance to 'decode_times'.
        Older xarray: fall back to legacy 'use_cftime' kwarg.
        """
        # Newer API (preferred)
        try:
            coder = xr.coders.CFDatetimeCoder(use_cftime="auto")
            return {"decode_times": coder}
        except Exception:
            pass
        # Fallback for older xarray
        return {"decode_times": True, "use_cftime": True}

    @staticmethod
    def _open_many(files: list[str], chunks: dict | None = None) -> xr.Dataset:
        opts = EnsembleMetricEvaluator._cf_decode_opts()
        return xr.open_mfdataset(
            files,
            combine="by_coords",
            mask_and_scale=True,
            parallel=False,
            chunks=chunks,
            **opts,
        )

    @classmethod
    def open_obs_series(
        cls,
        obs_root: str,
        product: str,
        period: str,
        try_vars=None,
        allow_missing_months: bool = True,
        chunks: dict | None = None,
    ) -> xr.Dataset:
        obs_root = Path(obs_root)
        y0, m0, y1, m1 = cls._parse_period_ym(period)

        files = []
        for y in range(y0, y1 + 1):
            m_start = m0 if y == y0 else 1
            m_end = m1 if y == y1 else 12
            for mm in range(m_start, m_end + 1):
                f = obs_root / f"{product}.{y}.{mm:02d}.nc"
                if f.exists():
                    files.append(str(f))
        if not files:
            raise FileNotFoundError(f"No obs files for {product} in {obs_root} over {period}")

        ds = cls._open_many(files, chunks=chunks)
        ds = cls._normalize_coords(ds)
        ds = cls._select_period_by_ym(ds, y0, m0, y1, m1, allow_missing_months)

        if try_vars:
            for v in try_vars:
                if v in ds.data_vars:
                    return ds[[v]]
            raise KeyError(f"None of {try_vars} found in {product} (have {list(ds.data_vars)})")
        return ds

    @classmethod
    def open_model_cube(
        cls,
        exp_root: str,
        var: str,
        nens: int,
        period: str,
        allow_missing_months: bool = True,
        chunks: dict | None = None,
    ) -> xr.DataArray:
        exp_root = Path(exp_root)
        y0, m0, y1, m1 = cls._parse_period_ym(period)

        members = []
        for en in range(1, nens + 1):
            tag = f"EN{en:02d}"
            member_files = sorted(exp_root.glob(f"{var}.{tag}.*.nc"))
            if not member_files:
                print(f"[WARN] No files for {var}.{tag} in {exp_root}")
                continue

            ds = cls._open_many([str(p) for p in member_files], chunks=chunks)
            ds = cls._normalize_coords(ds)

            if var not in ds:
                raise KeyError(f"{var} not found in {member_files[0].name}")

            ds = cls._select_period_by_ym(ds, y0, m0, y1, m1, allow_missing_months)
            da = ds[var].expand_dims({"ens": [tag]})
            members.append(da)

        if not members:
            raise FileNotFoundError(f"No model members loaded for {var} in {exp_root}")

        return xr.concat(members, dim="ens").sortby(["ens", "time"])

    @staticmethod
    def convert_units(da: xr.DataArray, varname: str, target_unit: str, product: str = 'e3sm') -> xr.DataArray:
        v = varname.upper()
        out = da.copy()

        # TS/TREFHT: K -> °C
        if target_unit in ["$^o$C", "°C"] and v in {"TS", "TREFHT"}:
            out = out - 273.15
            out.attrs["units"] = "°C"
            return out

        # PSL: Pa -> hPa
        if v == "PSL" and target_unit.lower().startswith("hpa"):
            try:
                if float(out.max().values) > 2_000.0:  # likely Pa
                    out = out * 0.01
            except Exception:
                pass
            out.attrs["units"] = "hPa"
            return out

        # PRECT: m/s or kg m-2 s-1 -> mm/day
        if product != "GPCP":
            if v == "PRECT" and "mm day" in target_unit:
                out = out * (86400.0 * 1000.0)
                out.attrs["units"] = "mm day$^{-1}$"
                return out

        # Fluxes: assume W m-2 already
        out.attrs["units"] = target_unit
        return out

    # --------------------------- core metrics ---------------------------

    def __init__(
        self,
        obs,
        model_dict,
        component,
        ref_dataset,
        var,
        vunit,
        output_path,
        n_boot_map: int = 500,
        n_boot_scalar: int = 1000,
        alpha: float = 0.05,
    ):
        self.obs_data = obs
        self.model_dict = model_dict
        self.component = component
        self.reference = ref_dataset
        self.var_name = var
        self.var_unit = vunit
        self.output_path = output_path
        self.n_boot_map = n_boot_map
        self.n_boot_scalar = n_boot_scalar
        self.alpha = alpha

        self.results = self.derive_metrics_data()

    # ---- utilities

    def _lat_name(self, da) -> str:
        if "lat" in da.coords: return "lat"
        if "latitude" in da.coords: return "latitude"
        raise ValueError("Latitude coordinate not found (lat|latitude).")

    def _generate_coslat_weight(self, da: xr.DataArray) -> xr.DataArray:
        lat_name = self._lat_name(da)
        w1d = np.cos(np.deg2rad(da[lat_name]))
        if lat_name not in da.dims:
            w1d = w1d.expand_dims(dim=lat_name)
        w, _ = xr.broadcast(w1d, da)
        return w

    def _get_ensemble_dim(self, da: xr.DataArray) -> str:
        for dim in da.dims:
            if dim in {"ens", "ensemble", "member"}:
                return dim
        for dim in da.dims:
            if dim not in {"time", "lat", "lon", "latitude", "longitude"} and da.sizes[dim] <= 200:
                return dim
        raise ValueError("No ensemble dimension found (ens|ensemble|member).")

    def unify_fill_missing_da(self, da: xr.DataArray, sentinels=None) -> xr.DataArray:
        out = da.copy()
        for s in (sentinels or ()):
            try:
                out = out.where(out != s, np.nan)
            except Exception:
                pass
        for k in ("missing_value", "_FillValue"):
            v = out.attrs.get(k)
            if v is not None:
                try:
                    out = out.where(out != v, np.nan)
                except Exception:
                    pass
            out.attrs.pop(k, None)
            out.encoding.pop(k, None)
        if np.issubdtype(out.dtype, np.floating):
            out = out.astype("float32")
        return out

    def _ensure_single_chunk(self, da: xr.DataArray, core_dims: list[str]) -> xr.DataArray:
        if not hasattr(da.data, "chunks"):
            return da
        chunk_map = {d: -1 for d in core_dims if d in da.dims}
        return da.chunk(chunk_map)

    def _strip_nondim_coords(self, da: xr.DataArray) -> xr.DataArray:
        """
        Drop all non-dimension coordinates (e.g., scalar 'quantile', 'boot')
        to avoid Dataset merge conflicts.
        """
        if not isinstance(da, xr.DataArray):
            return da
        drop_these = [c for c in da.coords if c not in da.dims]
        if drop_these:
            da = da.reset_coords(drop_these, drop=True)
        return da

    # ---- per-gridcell map metrics

    def bias_map_fn(self, model, obs, ens_dim="ens"):
        if ens_dim in obs.dims:
            obs = obs.mean(dim=ens_dim, skipna=True)
        latn = self._lat_name(model)
        model = model.transpose(ens_dim, latn, "lon")
        obs   = obs.transpose(latn, "lon")
        return model.mean(dim=ens_dim, skipna=True) - obs

    def rmse_map_fn(self, model, obs, ens_dim="ens"):
        latn = self._lat_name(model)
        model = model.transpose(ens_dim, latn, "lon")
        if ens_dim in obs.dims:
            obs = obs.mean(dim=ens_dim, skipna=True)
        obs = obs.expand_dims({ens_dim: model[ens_dim]}).transpose(ens_dim, latn, "lon")
        return xs.rmse(model, obs, dim=ens_dim, skipna=True)

    def spread_map_fn(self, model, obs=None, ens_dim="ens"):
        latn = self._lat_name(model)
        model = model.transpose(ens_dim, latn, "lon")
        return model.std(dim=ens_dim, skipna=True)
    
    def crps_map_fn(self, model, obs, ens_dim="ens"):
        # Strip accidental ensemble on obs
        if ens_dim in obs.dims:
            obs = obs.mean(dim=ens_dim, skipna=True)

        latn = self._lat_name(model)
        model = model.transpose(ens_dim, latn, "lon")
        obs   = obs.transpose(latn, "lon")

        model = self._ensure_single_chunk(model, [ens_dim])

        # xskillscore >= 0.0.24 style (no dask kwargs, no dim=[])
        try:
            crps = xs.crps_ensemble(obs, model, member_dim=ens_dim)
        except TypeError:
            # very old xskillscore fallback (if available in your env)
            crps = xs.crps_ensemble(obs, model, member_dim=ens_dim)  # keep same; older also accepts this

        return crps.where(obs.notnull())

    # ---- bootstrap wrappers
    def bootstrap_ci_map(
        self, model, obs, metric_fn, ens_dim="ens", n_boot=1000, alpha=0.05
    ):
        """Bootstrap CIs for a map-valued metric across the ensemble members."""
        model = self._ensure_single_chunk(model, [ens_dim])

        rng   = np.random.default_rng(seed=42)
        n_ens = model.sizes[ens_dim]
        boots = []

        for _ in range(n_boot):
            idx = rng.integers(0, n_ens, size=n_ens)
            res  = model.isel({ens_dim: idx})
            res  = self._ensure_single_chunk(res, [ens_dim])
            boots.append(metric_fn(res, obs, ens_dim=ens_dim))

        stack = xr.concat(boots, dim="boot")

        lower = stack.quantile(alpha / 2, dim="boot")
        upper = stack.quantile(1 - alpha / 2, dim="boot")

        mean_metric = metric_fn(model, obs, ens_dim=ens_dim)
        signif = (mean_metric < lower) | (mean_metric > upper)

        # scrub stray non-dim coords early
        return (
            self._strip_nondim_coords(lower),
            self._strip_nondim_coords(upper),
            self._strip_nondim_coords(signif),
        )

    def compute_ci_and_significance(
        self, metric_name, metric_fn, model, obs, ens_dim="ens",
        alpha=0.05, n_boot=1000
    ):
        try:
            lo, hi, mask = self.bootstrap_ci_map(
                model, obs, metric_fn, ens_dim=ens_dim, n_boot=n_boot, alpha=alpha
            )
            # name & attrs
            mask.name = f"{metric_name}_significance_mask"
            lo.name   = f"{metric_name}_ci_lower"
            hi.name   = f"{metric_name}_ci_upper"
            mask.attrs.update({"units": "1", "long_name": f"Significance mask for {metric_name.replace('_',' ')}"})
            lo.attrs.update({"units": self.var_unit, "long_name": f"Lower 95% CI for {metric_name.replace('_',' ')}"})
            hi.attrs.update({"units": self.var_unit, "long_name": f"Upper 95% CI for {metric_name.replace('_',' ')}"})
            return lo, hi, mask
        except Exception as e:
            print(f"[WARN] CI/significance for {metric_name} failed: {e}")
            return None, None, None

    def bootstrap_confidence_interval(self, metric_fn, members, obs, ens_dim="ens", n_boot=None, alpha=None):
        rng = np.random.default_rng(seed=42)
        n_boot = n_boot or self.n_boot_scalar
        alpha = self.alpha if alpha is None else alpha
        n_ens = members.sizes[ens_dim]
        vals = []

        for _ in range(n_boot):
            idx = rng.integers(0, n_ens, size=n_ens)
            sample = members.isel({ens_dim: idx})
            try:
                v = metric_fn(sample.mean(dim=ens_dim), obs, ens_dim=ens_dim)
                arr = np.asarray(v)
                vfloat = float(arr) if arr.size == 1 else float(np.nanmean(arr))
                if np.isfinite(vfloat):
                    vals.append(vfloat)
            except Exception:
                continue

        if not vals:
            return np.nan, np.nan, np.nan

        vals = np.array(vals, dtype=float)
        return (float(np.percentile(vals, 100 * alpha / 2)),
                float(np.percentile(vals, 100 * (1 - alpha / 2))),
                float(vals.mean()))

    # ---- additional diagnostics

    def compute_rank_histogram(self, members: xr.DataArray, obs: xr.DataArray, normalize=False):
        ens_sorted = np.sort(members.values, axis=0)
        ranks = np.sum(ens_sorted < obs.values, axis=0).flatten()
        n_ens = ens_sorted.shape[0]
        valid = ranks[(~np.isnan(ranks)) & (ranks >= 0) & (ranks <= n_ens)]
        hist, _ = np.histogram(valid, bins=np.arange(n_ens + 2) - 0.5, density=normalize)
        return xr.DataArray(hist, dims=["rank"], coords={"rank": np.arange(n_ens + 1)})

    def _as_float(self, da: xr.DataArray) -> float:
        try:
            return float(da.item())
        except Exception:
            try:
                return float(da.compute().item())
            except Exception:
                v = da.values
                return float(v.item() if np.ndim(v) == 0 else np.nanmean(v))

    def compute_ensemble_coverage(self, ens_dim: str, members: xr.DataArray, obs: xr.DataArray):
        ens_min = members.min(dim=ens_dim)
        ens_max = members.max(dim=ens_dim)
        within = ((obs >= ens_min) & (obs <= ens_max)).astype(float)
        mean_da = within.mean()
        meanv = self._as_float(mean_da)
        return xr.DataArray(
            meanv,
            attrs={"units": "1", "long_name": "Fraction of domain where obs within ensemble envelope"},
        )

    def safe_mean(self, val):
        if isinstance(val, xr.DataArray):
            try:
                return self._as_float(val.mean(skipna=True))
            except Exception:
                return np.nan
        if isinstance(val, (int, float, np.number)):
            return float(val)
        return np.nan

    # ---- composite metric derivation

    def _derive_hor_metrics_data(self, dso1, dsm1, ens_dim):
        # chunk safely (no new dims)
        dsm1 = dsm1.chunk({d: -1 for d in dsm1.dims if d in ["time", ens_dim, "lat", "lon"]})
        dso1 = dso1.chunk({d: -1 for d in dso1.dims if d in ["time", "lat", "lon"]})

        mec = {}
        bias_map = std_map = rmse_map = spread_map = mean_map = None

        ens_mean = dsm1.mean(dim=ens_dim, skipna=True)
        ens_std  = dsm1.std(dim=ens_dim,  skipna=True)

        mean_map = ens_mean.mean(dim="time", skipna=True) if "time" in ens_mean.dims else ens_mean
        weights = self._generate_coslat_weight(mean_map)

        # model global mean/RMS
        mec["mod_mean"] = (mean_map * weights).sum(dim=["lat", "lon"], skipna=True) / weights.sum(dim=["lat", "lon"], skipna=True)
        mec["mod_rms"]  = np.sqrt((((mean_map - mec["mod_mean"]) ** 2) * weights).sum(dim=["lat", "lon"], skipna=True) /
                                  weights.sum(dim=["lat", "lon"], skipna=True))

        if isinstance(dso1, xr.DataArray):
            dso_mean = dso1.mean(dim="time", skipna=True) if "time" in dso1.dims else dso1
            mec["obs_mean"] = (dso_mean * weights).sum(dim=["lat", "lon"], skipna=True) / weights.sum(dim=["lat", "lon"], skipna=True)
            mec["obs_rms"]  = np.sqrt((((dso_mean - mec["obs_mean"]) ** 2) * weights).sum(dim=["lat", "lon"], skipna=True) /
                                      weights.sum(dim=["lat", "lon"], skipna=True))

            mec["rmse_glb"] = xs.rmse(dso_mean, mean_map, dim=["lat", "lon"], weights=weights, skipna=True)
            mec["pcor"]     = xs.pearson_r(dso_mean - mec["obs_mean"], mean_map - mec["mod_mean"],
                                           dim=["lat", "lon"], weights=weights, skipna=True)

            diff     = ens_mean - dso1
            bias_map = diff.mean(dim="time", skipna=True) if "time" in diff.dims else diff
            mec["bias"] = (bias_map * weights).sum(dim=["lat", "lon"], skipna=True) / weights.sum(dim=["lat", "lon"], skipna=True)

            std_map = diff.std(dim="time",  skipna=True) if "time" in diff.dims else xr.zeros_like(diff)
            spread  = ens_std.mean(dim="time", skipna=True) if "time" in ens_std.dims else ens_std
            mec["spread"] = (spread * weights).sum(dim=["lat", "lon"], skipna=True) / weights.sum(dim=["lat", "lon"], skipna=True)

            dso_exp  = dso1.expand_dims({ens_dim: dsm1[ens_dim]})
            rerr     = xs.rmse(dsm1, dso_exp, dim=ens_dim, skipna=True)
            rmse_map = rerr.mean(dim="time", skipna=True) if "time" in rerr.dims else rerr
            mec["rmse"] = (rmse_map * weights).sum(dim=["lat", "lon"], skipna=True) / weights.sum(dim=["lat", "lon"], skipna=True)

        return mec, bias_map, std_map, rmse_map, spread, mean_map

    def derive_metrics_data(self):
        results = {}

        for model in self.model_dict:
            print(f"[INFO] Processing {model}")

            ens_dim  = self._get_ensemble_dim(self.model_dict[model])
            mod_full = self.model_dict[model]
            obs_full = self.obs_data

            # unify missing values
            DEFAULT_SENTINELS = (-9999.0, -999.0, 1.0e20, 9.96921e36)
            mod_full = self.unify_fill_missing_da(mod_full, sentinels=DEFAULT_SENTINELS)
            obs_full = self.unify_fill_missing_da(obs_full, sentinels=DEFAULT_SENTINELS)

            # drop degenerate time
            if "time" in mod_full.dims and mod_full.sizes["time"] == 1:
                mod_full = mod_full.squeeze("time")
            if "time" in obs_full.dims and obs_full.sizes["time"] == 1:
                obs_full = obs_full.squeeze("time")

            metrics, bias_map, bias_std_map, rmse_map, spread_map, mean_map = self._derive_hor_metrics_data(
                obs_full, mod_full, ens_dim
            )

            # derived scalars
            mse       = (metrics["rmse"] ** 2) if np.isfinite(self.safe_mean(metrics.get("rmse"))) else np.nan
            bias_sq   = (metrics["bias"] ** 2) if np.isfinite(self.safe_mean(metrics.get("bias"))) else np.nan
            spread_sq = (metrics["spread"] ** 2) if np.isfinite(self.safe_mean(metrics.get("spread"))) else np.nan
            sem       = metrics["spread"] / np.sqrt(mod_full.sizes[ens_dim]) if np.isfinite(self.safe_mean(metrics.get("spread"))) else np.nan
            metrics.update({
                "ssr": metrics["spread"] / metrics["rmse"] if np.isfinite(self.safe_mean(metrics.get("rmse"))) else np.nan,
                "bias_squared":  bias_sq,
                "spread_squared":spread_sq,
                "mse": mse,
                "sem": sem
            })

            # additional diagnostics
            rank_hist = self.compute_rank_histogram(mod_full, obs_full)
            coverage  = self.compute_ensemble_coverage(ens_dim, mod_full, obs_full)
            crps_map = self.compute_crps(mod_full, obs_full)
            w_crps    = self._generate_coslat_weight(crps_map) if isinstance(crps_map, xr.DataArray) else None
            crps_mean = ((crps_map * w_crps).sum(dim=["lat", "lon"], skipna=True) / w_crps.sum(dim=["lat", "lon"], skipna=True)
                         if w_crps is not None else np.nan)
            metrics.update({"rank_histogram": rank_hist, "coverage": coverage, "crps_mean": crps_mean})

            # scalar bootstraps
            bias_fn   = lambda m, o, ens_dim="ens": float((m - o).mean(dim=["lat", "lon"], skipna=True).values)
            rmse_fn   = lambda m, o, ens_dim="ens": float(xs.rmse(o, m, dim=["lat", "lon"], skipna=True).values)
            spread_fn = lambda m, o, ens_dim="ens": float(m.std(dim=ens_dim, skipna=True).mean(dim=["lat", "lon"], skipna=True).values)
            pcor_fn   = lambda m, o, ens_dim="ens": float(xs.pearson_r(o, m, dim=["lat", "lon"], skipna=True).values)
            ssr_fn    = lambda m, o, ens_dim="ens": spread_fn(m, o, ens_dim) / rmse_fn(m, o, ens_dim)

            try:
                bias_lb,   bias_ub,   _ = self.bootstrap_confidence_interval(bias_fn,   mod_full, obs_full, ens_dim)
                rmse_lb,   rmse_ub,   _ = self.bootstrap_confidence_interval(rmse_fn,   mod_full, obs_full, ens_dim)
                spread_lb, spread_ub, _ = self.bootstrap_confidence_interval(spread_fn, mod_full, obs_full, ens_dim)
                pcor_lb,   pcor_ub,   _ = self.bootstrap_confidence_interval(pcor_fn,   mod_full, obs_full, ens_dim)
                ssr_lb,    ssr_ub,    _ = self.bootstrap_confidence_interval(ssr_fn,    mod_full, obs_full, ens_dim)
                metrics.update({
                    "bias_ci_lower":   bias_lb,   "bias_ci_upper":   bias_ub,
                    "rmse_ci_lower":   rmse_lb,   "rmse_ci_upper":   rmse_ub,
                    "spread_ci_lower": spread_lb, "spread_ci_upper": spread_ub,
                    "pcor_ci_lower":   pcor_lb,   "pcor_ci_upper":   pcor_ub,
                    "ssr_ci_lower":    ssr_lb,    "ssr_ci_upper":    ssr_ub
                })
            except Exception as e:
                print(f"[WARN] Scalar bootstrap CI failed: {e}")

            # map-level bootstrap CIs/significance
            bias_ci_lo,  bias_ci_hi,  bias_ci_mask   = self.compute_ci_and_significance(
                "bias_map",  self.bias_map_fn,  mod_full, obs_full, ens_dim, alpha=self.alpha, n_boot=self.n_boot_map
            )
            rmse_ci_lo,  rmse_ci_hi,  rmse_ci_mask   = self.compute_ci_and_significance(
                "rmse_map",  self.rmse_map_fn,  mod_full, obs_full, ens_dim, alpha=self.alpha, n_boot=self.n_boot_map
            )
            spread_ci_lo,spread_ci_hi,spread_ci_mask = self.compute_ci_and_significance(
                "spread_map",self.spread_map_fn, mod_full, obs_full, ens_dim, alpha=self.alpha, n_boot=self.n_boot_map
            )
            crps_ci_lo,  crps_ci_hi,  crps_ci_mask   = self.compute_ci_and_significance(
                "crps_map",  self.crps_map_fn,  mod_full, obs_full, ens_dim, alpha=self.alpha, n_boot=self.n_boot_map
            )

            results[model] = {
                "metrics": metrics,
                "mod_map":  mod_full.mean(dim="time", skipna=True) if "time" in mod_full.dims else mod_full,
                "obs_map":  obs_full.mean(dim="time", skipna=True) if "time" in obs_full.dims else obs_full,
                "mean_map": mean_map,
                "spread_map": spread_map,
                "bias_map": bias_map,
                "bias_stddev_map": bias_std_map,
                "rmse_map": rmse_map,
                "crps_map": crps_map,
                "bias_map_ci_lower": bias_ci_lo,   "bias_map_ci_upper": bias_ci_hi,   "bias_map_significance_mask": bias_ci_mask,
                "rmse_map_ci_lower": rmse_ci_lo,   "rmse_map_ci_upper": rmse_ci_hi,   "rmse_map_significance_mask": rmse_ci_mask,
                "spread_map_ci_lower": spread_ci_lo,"spread_map_ci_upper": spread_ci_hi,"spread_map_significance_mask": spread_ci_mask,
                "crps_map_ci_lower": crps_ci_lo,   "crps_map_ci_upper": crps_ci_hi,   "crps_map_significance_mask": crps_ci_mask
            }

            # Save
            nc_path  = os.path.join(self.output_path, f"{model}_{self.var_name}_{self.reference}_ensemble_metrics.nc")
            csv_path = os.path.join(self.output_path, f"{model}_{self.var_name}_{self.reference}_ensemble_summary.csv")
            self.save_to_netcdf(nc_path, model, results[model])
            self.save_summary_csv(csv_path, model, results[model])

        return results

    # ------------------------------ I/O ------------------------------

    def save_to_netcdf(self, filepath, model, model_data):
        meta = {
            "mod_map":  {"units": self.var_unit, "long_name": f"{self.var_name} ensemble mean (model)"},
            "obs_map":  {"units": self.var_unit, "long_name": f"{self.var_name} mean ({self.reference})"},
            "rmse_map": {"units": self.var_unit, "long_name": f"{self.var_name} ensemble RMSE (model vs {self.reference})"},
            "bias_map": {"units": self.var_unit, "long_name": f"{self.var_name} ensemble mean bias (model - {self.reference})"},
            "spread_map": {"units": self.var_unit, "long_name": f"{self.var_name} ensemble spread"},
            "mean_map": {"units": self.var_unit, "long_name": f"{self.var_name} ensemble mean"},
            "bias_stddev_map": {"units": self.var_unit, "long_name": f"{self.var_name} stddev of (ens-obs) over time"},
            "bias_map_ci_lower": {"units": self.var_unit, "long_name": "Lower 95% CI for bias map"},
            "bias_map_ci_upper": {"units": self.var_unit, "long_name": "Upper 95% CI for bias map"},
            "bias_map_significance_mask": {"units": "1", "long_name": "Significance mask for bias map"},
            "rmse_map_ci_lower": {"units": self.var_unit, "long_name": "Lower 95% CI for RMSE map"},
            "rmse_map_ci_upper": {"units": self.var_unit, "long_name": "Upper 95% CI for RMSE map"},
            "rmse_map_significance_mask": {"units": "1", "long_name": "Significance mask for RMSE map"},
            "crps_map_ci_lower": {"units": self.var_unit, "long_name": "Lower 95% CI for CRPS map"},
            "crps_map_ci_upper": {"units": self.var_unit, "long_name": "Upper 95% CI for CRPS map"},
            "crps_map_significance_mask": {"units": "1", "long_name": "Significance mask for CRPS map"},
            "spread_map_ci_lower": {"units": self.var_unit, "long_name": "Lower 95% CI for spread map"},
            "spread_map_ci_upper": {"units": self.var_unit, "long_name": "Upper 95% CI for spread map"},
            "spread_map_significance_mask": {"units": "1", "long_name": "Significance mask for spread map"},
            # scalars
            "mod_mean": {"units": self.var_unit, "long_name": f"{self.var_name} area-weighted mean (model)"},
            "mod_rms":  {"units": self.var_unit, "long_name": f"{self.var_name} global RMS (model)"},
            "obs_mean": {"units": self.var_unit, "long_name": f"{self.var_name} area-weighted mean ({self.reference})"},
            "obs_rms":  {"units": self.var_unit, "long_name": f"{self.var_name} global RMS ({self.reference})"},
            "rmse": {"units": self.var_unit, "long_name": f"{self.var_name} spatial-mean ensemble RMSE"},
            "rmse_glb": {"units": self.var_unit, "long_name": f"{self.var_name} ensemble mean global RMSE"},
            "spread": {"units": self.var_unit, "long_name": f"{self.var_name} spatial-mean ensemble spread"},
            "ssr": {"units": "1", "long_name": "Spread-to-RMSE ratio"},
            "bias": {"units": self.var_unit, "long_name": "Spatial mean ensemble mean bias"},
            "bias_squared": {"units": f"{self.var_unit}^2", "long_name": "Squared ensemble mean bias"},
            "spread_squared": {"units": f"{self.var_unit}^2", "long_name": "Squared ensemble spread"},
            "mse": {"units": f"{self.var_unit}^2", "long_name": "Mean square error (bias² + spread²)"},
            "sem": {"units": self.var_unit, "long_name": "Standard error of the ensemble mean"},
            "bias_ci_lower": {"units": self.var_unit, "long_name": "Lower 95% CI for spatial-mean bias"},
            "bias_ci_upper": {"units": self.var_unit, "long_name": "Upper 95% CI for spatial-mean bias"},
            "rmse_ci_lower": {"units": self.var_unit, "long_name": "Lower 95% CI for spatial-mean RMSE"},
            "rmse_ci_upper": {"units": self.var_unit, "long_name": "Upper 95% CI for spatial-mean RMSE"},
            "spread_ci_lower": {"units": self.var_unit, "long_name": "Lower 95% CI for spatial-mean spread"},
            "spread_ci_upper": {"units": self.var_unit, "long_name": "Upper 95% CI for spatial-mean spread"},
            "pcor": {"units": "1", "long_name": "Pattern correlation (anomalies)"},
            "pcor_ci_lower": {"units": "1", "long_name": "Lower 95% CI for pattern correlation"},
            "pcor_ci_upper": {"units": "1", "long_name": "Upper 95% CI for pattern correlation"},
            "ssr_ci_lower": {"units": "1", "long_name": "Lower 95% CI for SSR"},
            "ssr_ci_upper": {"units": "1", "long_name": "Upper 95% CI for SSR"},
            "rank_histogram": {"units": "count", "long_name": "Rank histogram of obs within ensemble"},
            "coverage": {"units": "1", "long_name": "Fraction of domain where obs within ensemble envelope"},
            "crps_map": {"units": self.var_unit, "long_name": f"CRPS map for {self.var_name}"},
            "crps_mean": {"units": self.var_unit, "long_name": f"Spatial mean CRPS for {self.var_name}"},
        }

        ds_out = {}

        # metrics (scalars/small arrays)
        for key, val in model_data["metrics"].items():
            da = val if isinstance(val, xr.DataArray) else xr.DataArray(val)
            if "time" in getattr(da, "coords", {}) and "time" not in getattr(da, "dims", []):
                da = da.reset_coords("time", drop=True)
            da = self._strip_nondim_coords(da)
            if isinstance(da, xr.DataArray) and da.ndim == 0:
                da = da.expand_dims({"scalar_dim": [0]})
            da.attrs.update(meta.get(key, {}))
            ds_out[key] = da

        # maps & CIs
        for key, val in model_data.items():
            if isinstance(val, xr.DataArray):
                da = val
                if "time" in da.coords and "time" not in da.dims:
                    da = da.reset_coords("time", drop=True)
                da = self._strip_nondim_coords(da)
                da.attrs.update(meta.get(key, {}))
                ds_out[key] = da

        ds = xr.Dataset(ds_out)
        ds.attrs.update({
            "model": model,
            "component": self.component,
            "reference_dataset": self.reference,
            "variable": self.var_name,
            "units": self.var_unit,
            "created_by": "EnsembleMetricEvaluator",
            "creation_time": datetime.now().isoformat()
        })
        ds.to_netcdf(filepath)
        print(f"[INFO] Saved metrics for {model} to {filepath}")

    def save_summary_csv(self, filepath, model, model_data):
        m = model_data["metrics"]
        row = {
            "model": model,
            "RMSE":           self.safe_mean(m.get("rmse")),
            "RMSE_GLB":       self.safe_mean(m.get("rmse_glb")),
            "PCOR":           self.safe_mean(m.get("pcor")),
            "Spread":         self.safe_mean(m.get("spread")),
            "SSR":            self.safe_mean(m.get("ssr")),
            "Bias^2":         self.safe_mean(m.get("bias_squared")),
            "Spread^2":       self.safe_mean(m.get("spread_squared")),
            "MSE":            self.safe_mean(m.get("mse")),
            "SEM":            self.safe_mean(m.get("sem")),
            "Coverage":       self.safe_mean(m.get("coverage")),
            "CRPS":           self.safe_mean(m.get("crps_mean")),
            "Bias_CI_Lower":  m.get("bias_ci_lower"),
            "Bias_CI_Upper":  m.get("bias_ci_upper"),
            "RMSE_CI_Lower":  m.get("rmse_ci_lower"),
            "RMSE_CI_Upper":  m.get("rmse_ci_upper"),
            "Spread_CI_Lower":m.get("spread_ci_lower"),
            "Spread_CI_Upper":m.get("spread_ci_upper"),
            "PCOR_CI_Lower":  m.get("pcor_ci_lower"),
            "PCOR_CI_Upper":  m.get("pcor_ci_upper"),
            "SSR_CI_Lower":   m.get("ssr_ci_lower"),
            "SSR_CI_Upper":   m.get("ssr_ci_upper"),
        }
        if all(pd.isna(v) for k, v in row.items() if k != "model"):
            print(f"[WARN] No valid metrics found for {model}, CSV not written.")
            return
        df = pd.DataFrame([row])
        if os.path.exists(filepath):
            os.remove(filepath)
        df.to_csv(filepath, index=False)
        print(f"[INFO] Saved summary CSV to {filepath}")

    # ---- CRPS convenience

    def compute_crps(self, members: xr.DataArray, obs: xr.DataArray) -> xr.DataArray:
        try:
            ens_dim = self._get_ensemble_dim(members)
            latn = self._lat_name(members)
            members = members.transpose(ens_dim, latn, "lon")
            obs     = obs.transpose(latn, "lon")
            members = self._ensure_single_chunk(members, [ens_dim])
            crps = xs.crps_ensemble(obs, members, member_dim=ens_dim, dim=[])
            return crps.where(obs.notnull())
        except Exception as e:
            print(f"[WARN] CRPS computation failed: {e}")
            return xr.full_like(obs, np.nan)


In [4]:
if __name__ == "__main__":
    # --- paths ---
    data_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_output/analysis_bias"
    os.makedirs(out_path, exist_ok=True)

    freq = "monthly"
    component = "atm"
    run_seg = "da"
    obs_path = f"{data_path}/obs_data/{freq}"

    # Build experiments
    exp_dict: Dict[str, dict] = extract_exp_info(data_path)

    # pretty labels (optional; not used for indexing)
    model_label_map = {
        "CTRL10-S0": "CTRL (ENS=10)",
        "DART10-S0": "EAM-DART (ENS=10)",
        "DART20-S0": "EAM-DART (ENS=20)",
        "DART40-S0": "EAM-DART (ENS=40)",
    }

    # --- variables ---
    var_dict = {
        "FLUT": {
            "alias": "FLUT", "unit": "W m$^{-2}$",
            "level": np.linspace(-10, 10, 51), "comp": "atm", "ref": "CERES-OAFlux",
        },
        "PRECT": {
            "alias": "PRECT", "unit": "mm day$^{-1}$",
            "level": np.linspace(-2, 2, 51), "comp": "atm", "ref": "GPCP",
        },
        "TMQ": {
            "alias": "TMQ", "unit": "kg m$^{-2}$",
            "level": np.linspace(-10, 10, 51), "comp": "atm", "ref": "ERA5",
        },
        "TS": {
            "alias": "TS", "unit": "°C",
            "level": np.linspace(-8, 8, 51), "comp": "atm", "ref": "ERA5",
        },
        "TREFHT": {
            "alias": "TREFHT", "unit": "°C",
            "level": np.linspace(-8, 8, 51), "comp": "atm", "ref": "ERA5",
        },
        "PSL": {
            "alias": "PSL", "unit": "hPa",
            "level": np.linspace(-10, 10, 51), "comp": "atm", "ref": "ERA5",
        },
    }

    obs_var_map = {
        "CERES-OAFlux": {"FLUT": ["FLUT", "toa_lw_up"], "FLNT": ["FLNT", "toa_lw_net"]},
        "GPCP": {"PRECT": ["PRECT", "pr", "precip"]},
        "ERA5": {
            "TMQ": ["TMQ", "tcwv"],
            "TS": ["TS", "t2m", "skin_temperature"],
            "TREFHT": ["TREFHT", "t2m"],
            "PSL": ["PSL", "msl"],
        },
    }

    for var in var_dict.keys():
        for exp_key, label in model_label_map.items():    
            var_cfg = var_dict[var]
            exp_cfg = exp_dict[exp_key]
            EnsembleMetricEvaluator.run_one(
                var_name=var,
                var_cfg=var_cfg,
                freq=freq,
                run_seg=run_seg,
                component=component,
                exp_key=exp_key,
                exp_meta=exp_cfg,
                obs_var_map=obs_var_map,
                model_label=label,
                obs_root=obs_path,
                out_path=out_path,
            )


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CLDTOT' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CRE' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CRE_SRF' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'EVAP' has multiple 

[INFO] Processing CTRL10-S0
[INFO] Saved metrics for CTRL10-S0 to /compyfs/zhan391/v3_dart_cda_scratch/diag_output/analysis_bias/CTRL10-S0_FLUT_CERES-OAFlux_ensemble_metrics.nc
[INFO] Saved summary CSV to /compyfs/zhan391/v3_dart_cda_scratch/diag_output/analysis_bias/CTRL10-S0_FLUT_CERES-OAFlux_ensemble_summary.csv
[DONE] CTRL (ENS=10) · FLUT · 201112-201112


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CLDTOT' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CRE' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CRE_SRF' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'EVAP' has multiple 

[INFO] Processing DART10-S0
[INFO] Saved metrics for DART10-S0 to /compyfs/zhan391/v3_dart_cda_scratch/diag_output/analysis_bias/DART10-S0_FLUT_CERES-OAFlux_ensemble_metrics.nc
[INFO] Saved summary CSV to /compyfs/zhan391/v3_dart_cda_scratch/diag_output/analysis_bias/DART10-S0_FLUT_CERES-OAFlux_ensemble_summary.csv
[DONE] EAM-DART (ENS=10) · FLUT · 201112-201112


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CLDTOT' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CRE' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CRE_SRF' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'EVAP' has multiple 

[INFO] Processing DART20-S0
[INFO] Saved metrics for DART20-S0 to /compyfs/zhan391/v3_dart_cda_scratch/diag_output/analysis_bias/DART20-S0_FLUT_CERES-OAFlux_ensemble_metrics.nc
[INFO] Saved summary CSV to /compyfs/zhan391/v3_dart_cda_scratch/diag_output/analysis_bias/DART20-S0_FLUT_CERES-OAFlux_ensemble_summary.csv
[DONE] EAM-DART (ENS=20) · FLUT · 201112-201112


/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CLDTOT' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CRE' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'CRE_SRF' has multiple fill values {np.float32(-999.0), np.float32(-9999.0)} defined, decoding all values to NaN.
  var = coder.decode(var, name=name)
/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/xarray/conventions.py:204: SerializationWarning: variable 'EVAP' has multiple 

[INFO] Processing DART40-S0
[INFO] Saved metrics for DART40-S0 to /compyfs/zhan391/v3_dart_cda_scratch/diag_output/analysis_bias/DART40-S0_FLUT_CERES-OAFlux_ensemble_metrics.nc
[INFO] Saved summary CSV to /compyfs/zhan391/v3_dart_cda_scratch/diag_output/analysis_bias/DART40-S0_FLUT_CERES-OAFlux_ensemble_summary.csv
[DONE] EAM-DART (ENS=40) · FLUT · 201112-201112
[INFO] Processing CTRL10-S0
[INFO] Saved metrics for CTRL10-S0 to /compyfs/zhan391/v3_dart_cda_scratch/diag_output/analysis_bias/CTRL10-S0_PRECT_GPCP_ensemble_metrics.nc
[INFO] Saved summary CSV to /compyfs/zhan391/v3_dart_cda_scratch/diag_output/analysis_bias/CTRL10-S0_PRECT_GPCP_ensemble_summary.csv
[DONE] CTRL (ENS=10) · PRECT · 201112-201112
[INFO] Processing DART10-S0
[INFO] Saved metrics for DART10-S0 to /compyfs/zhan391/v3_dart_cda_scratch/diag_output/analysis_bias/DART10-S0_PRECT_GPCP_ensemble_metrics.nc
[INFO] Saved summary CSV to /compyfs/zhan391/v3_dart_cda_scratch/diag_output/analysis_bias/DART10-S0_PRECT_GPCP_ensem